# SIFLOW · run_1 · MDLM data prep

Tokenizes OpenWebText into length-256 GPT-2 chunks (train + a disjoint val split used as the MAUVE reference). Lower `N_TRAIN` for a quick smoke.

**Runtime:** comfortably under one Colab session.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
from transformers import AutoTokenizer
from siflow.data import build_token_chunks

tok = AutoTokenizer.from_pretrained("gpt2")
N_TRAIN = 200_000   # ~51M tokens (set 20_000 for a quick smoke)
N_VAL   = 5_000

print("train chunks:", build_token_chunks(tok, 256, N_TRAIN, f"{BASE}/data/owt_gpt2_256.npy",
      dataset="Skylion007/openwebtext", split="train"))
print("val chunks:",   build_token_chunks(tok, 256, N_VAL, f"{BASE}/data/owt_gpt2_val.npy",
      dataset="Skylion007/openwebtext", split="train", skip_seqs=N_TRAIN))

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_1_data.zip', include=['data/owt_gpt2_256.npy', 'data/owt_gpt2_val.npy'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)